In [1]:
import os
import sys
import json
import pathlib

from collections import namedtuple
from concurrent.futures import (
    ThreadPoolExecutor, 
    as_completed
)

from tqdm import tqdm
from loguru import logger

import pandas as pd
import numpy as np
import torch

In [2]:
# Add project root to path
sys.path.insert(
    0, 
    os.path.abspath(
        os.path.join(
            os.getcwd(), 
            '..'
        )
    )
)

from src import models as ml
from src import helper as hl

In [ ]:
# Define global variables
CFG_DATA_DIR = pathlib.Path('..')/'data'
CFG_FIGURES_DIR = CFG_DATA_DIR/'fig'
CFG_LOGFILE_DIR = CFG_DATA_DIR/'logs'
CFG_PARQUET_DIR = CFG_DATA_DIR/'parquet'
CFG_PICKLE_DIR = CFG_DATA_DIR/'pickle'

with (
    open(CFG_DATA_DIR / 'txt' / 'client_ids__serveo.txt', 'r') as fp_serveo, 
    open(CFG_DATA_DIR / 'txt' / 'client_ids__divvy.txt', 'r') as fp_divvy, 
    open(CFG_DATA_DIR / 'txt' / 'client_ids__citi.txt', 'r') as fp_citi, 
):
    CFG_CLIENT_IDS = {
        # Identifiers of Serveo dataset
        'serveo': json.load(fp_serveo),
        # Identifiers of Divvy dataset
        'divvy': json.load(fp_divvy),
        # Identifiers of Citi dataset
        'citi': json.load(fp_citi)
    }

# Experiment Parameters

In [4]:
FedGBDP_Experiment = namedtuple(
    'FedGBDP_Experiment',
    [
        'federation',
        'n_estimators',
        'parquet_bs',
        'bs',
        'num_rounds',
        'local_epochs',
        'early_stop',
        'patience',
        'conv_channels',
        'fc_layers',
        'dropout_rate',
        'mu',
        'fraction_fit',
        'fraction_eval',
    ]
)

In [5]:
args = FedGBDP_Experiment(
    federation='citi',
    n_estimators=37,
    bs=64,
    parquet_bs=4096,
    num_rounds=15,
    local_epochs=10,
    early_stop=True,
    patience=5,
    mu=0.125,
    conv_channels=32,
    fc_layers='',
    dropout_rate=0.13,
    fraction_fit=0.25,
    fraction_eval=0.25
)

# Load TEST dataset

In [6]:
df_test = pd.read_parquet( 
    path=(
        CFG_DATA_DIR / 
        args.federation / 
        'h6_w168_multi'
    ),
    engine='pyarrow',
    filters=[
        ('split', '==', 'test')
    ],
    memory_map=True
).reset_index(
    level=0,
    drop=True
)

# Local FedGBDP: Evaluate on the test set of each client

In [7]:
model_path = (
    CFG_DATA_DIR /
    'pth' /
    (
        'fedxgb_ver2026-01-21_10-40-49_citi_fraction_fit=0.25_fraction_eval=0.25_proximal_mu=0.125' if args.federation == 'citi' else (
            'fedxgb_ver2026-01-19_13-49-47_divvy_fraction_fit=0.25_fraction_eval=0.25_proximal_mu=0.125' if args.federation == 'divvy' else (
                'fedxgb_ver2026-01-27_09-33-31_serveo_fraction_fit=0.25_fraction_eval=0.25_proximal_mu=0.125' if args.federation == 'serveo' else None
            )
        )
    )
)

In [8]:
xgb_models = {
    k:v 
    for v,k
    in np.load(
        model_path / f'fedxgb_{args.federation}.flwr_global.epoch0.xgb_trees.npy',
        allow_pickle=True
    )
}

In [9]:
dataset_params__y_feats = ['target_demand_arrivals', 'target_demand_departures']

if args.federation == 'serveo':
    df_test.rename(
        {
            'target_demand_inbound': dataset_params__y_feats[0], 
            'target_demand_outbound': dataset_params__y_feats[1]
        },
        axis=1,
        inplace=True
    )

In [10]:
with tqdm() as pbar:
    with ThreadPoolExecutor(
        max_workers=128,
        thread_name_prefix=f'fedxgb_{args.federation}_'
    ) as tpe:
        futures = {}

        for station_id, station_data in df_test.groupby(level=0):
            futures[
                tpe.submit(
                    xgb_models[str(station_id).zfill(15)].predict, 
                    station_data.drop(
                        [
                            feat 
                            for feat
                            in station_data.columns
                            if 'target_demand' in feat
                        ],
                        axis=1
                    )
                )
            ] = station_id

        for future in as_completed(futures):
            station_id = futures[future]

            df_test.loc[
                pd.IndexSlice[
                    station_id,
                    :
                ],
                [
                    'target_demand_arrivals__local_xgb',
                    'target_demand_departures__local_xgb'
                ]
            ] = np.around(
                future.result(),
                decimals=0
            )

            pbar.update(1)

1497it [00:12, 121.71it/s]


# Global FedGBDP: Evaluate on the test set of each client

In [11]:
torch.set_num_threads(8)

In [12]:
global_model = ml.FedXGBllrCNN(
    num_clients=len(
        CFG_CLIENT_IDS[args.federation]
    ), 
    trees_per_client=args.n_estimators, 
    in_channels=2, 
    conv_channels=args.conv_channels, 
    fc_layers=args.fc_layers, 
    out_channels=2, 
    dropout_rate=args.dropout_rate
)

global_model_dict = torch.load(model_path / f'fedxgb_{args.federation}.flwr_global.epoch15.pth')

hl.set_parameters(
    global_model,
    global_model_dict['parameters']
)

global_model.eval()

FedXGBllrCNN(
  (conv1d): Conv1d(2, 32, kernel_size=(37,), stride=(37,))
  (relu): ReLU()
  (dropout): Dropout(p=0.13, inplace=False)
  (fc): Sequential(
    (0): Linear(in_features=47904, out_features=2, bias=True)
  )
)

In [13]:
def model_inference(client_X, total_trees, in_channels):    
    with torch.no_grad():
        # Main code 
        client_id = client_X.with_suffix('').with_suffix('').name.split('client')[1]
        y_pred = []

        logger.info(f'client #{client_id} - Loading test dataset (1D-CNN input)')
        X_tensor = torch.from_numpy(
            pd.read_parquet(
                client_X,
                engine='pyarrow'
            ).values
        )

        logger.info(f'client #{client_id} - Preparing 1D-CNN input')
        X_tensor = X_tensor.view(
            -1,  # number of records
            total_trees,  # total number of estimators
            in_channels  # number of features per estimator
        ).swapaxes(
            -2,
            -1
        )  # shape: [records, features, estimators]

        logger.info(f'client #{client_id} - Model inference (1D-CNN output)')
        y_pred = global_model(X_tensor) # Perform forward pass

        assert not torch.isnan(y_pred).any(), logger.error("Tensor contains NaNs!")

        return (
            y_pred.numpy(), 
            client_id
        )

In [ ]:
CFG_OID_DTYPE = int if args.federation == 'serveo' else str

def _worker_inference(client_X: pathlib.Path, total_trees: int, in_channels: int) -> None:    
    # Fetch results from completed job
    y_pred, client_id = model_inference(
        client_X,
        total_trees, 
        in_channels
    )

    # Add results to master table
    df_test.loc[
        pd.IndexSlice[
            CFG_OID_DTYPE(client_id),
            :
        ],
        [
            'target_demand_arrivals__global_fedxgb',
            'target_demand_departures__global_fedxgb'
        ]
    ] = y_pred

In [15]:
import gc


client_data = (
    CFG_DATA_DIR /
    'pth' / 
    f'fedxgb_{args.federation}'
)

Xs = sorted(
    client_data.glob(
        f'cnn_test_dataset__n_estimators_{args.n_estimators}.X.client*.parquet.gzip'
    )
)


with tqdm(
    total=len(Xs)
) as pbar:
    with ThreadPoolExecutor(
        max_workers=25
    ) as tpe:
        
        futures = {
            tpe.submit(
                _worker_inference, 
                *(
                    X,
                    global_model.total_trees, 
                    2
                )
            ): X
            for X 
            in Xs
        }

        for future in as_completed(futures):
            future.result()

            # Remove reference to completed job
            futures.pop(future)
            gc.collect()

            # Proceed with the next job
            pbar.update(1)

  0%|          | 0/1468 [00:00<?, ?it/s]2026-01-31 10:41:26.010 | INFO     | __main__:model_inference:7 - client #2733.03 - Loading test dataset (1D-CNN input)
2026-01-31 10:41:26.012 | INFO     | __main__:model_inference:7 - client #2782.02 - Loading test dataset (1D-CNN input)
2026-01-31 10:41:26.013 | INFO     | __main__:model_inference:7 - client #2832.03 - Loading test dataset (1D-CNN input)
2026-01-31 10:41:26.014 | INFO     | __main__:model_inference:7 - client #2883.03 - Loading test dataset (1D-CNN input)
2026-01-31 10:41:26.016 | INFO     | __main__:model_inference:7 - client #2912.08 - Loading test dataset (1D-CNN input)
2026-01-31 10:41:26.017 | INFO     | __main__:model_inference:7 - client #2932.01 - Loading test dataset (1D-CNN input)
2026-01-31 10:41:26.019 | INFO     | __main__:model_inference:7 - client #2951.05 - Loading test dataset (1D-CNN input)
2026-01-31 10:41:26.019 | INFO     | __main__:model_inference:7 - client #3007.05 - Loading test dataset (1D-CNN input)


# Since FL randomly selects clients for training / testing, some clients may not be selected at all for a phase (train / test). Let's add their corresponing forecasts manually...

In [17]:
nan_oids = df_test.loc[
    df_test.target_demand_departures__global_fedxgb.isna()
].index.get_level_values(0).unique()

In [18]:
logger.info(f'{nan_oids=}')

2026-01-31 12:41:12.401 | INFO     | __main__:<module>:1 - nan_oids=Index(['3460.02', '3564.04', '3665.06', '3771.06', '4110.10', '4128.08',
       '4620.02', '4721.01', '4748.07', '5230.02', '5270.08', '5414.06',
       '5433.03', '5669.12', '5779.1', '6131.12', '6659.1', '6743.06',
       '6839.1', '6925.09', '7121.02', '7382.04', '7522.02', '7625.18',
       '7903.02', '8366.07', '8532.03', '8715.01', '8732.04'],
      dtype='object', name='station_id')


In [19]:
dataset_params = dict(
    dataset_tag=args.federation,
    njobs=1,
    data_dir=CFG_DATA_DIR,
    figures_dir=CFG_FIGURES_DIR,
    parquet_dir=CFG_PARQUET_DIR,
    pickle_dir=CFG_PICKLE_DIR,
    y_feats=dataset_params__y_feats,
    bs=args.bs,
    parquet_bs=args.parquet_bs
)

for oid in nan_oids:
    hl.fedxgbllr_cnn_create_tensor_dataset(
        list(
            map(
                lambda l: [l[1], l[0]], 
                xgb_models.items()
            )
        ),
        {
            'X': df_test.xs(oid, level=0, drop_level=False).drop(
                [
                    *dataset_params__y_feats,
                    *[
                        f'{feat}__global_fedxgb'
                        for feat
                        in dataset_params__y_feats
                    ],
                    *[
                        f'{feat}__local_xgb'
                        for feat
                        in dataset_params__y_feats
                    ]
                ], 
                axis=1
            ),
            'y': df_test.xs(oid, level=0, drop_level=False).loc[:, dataset_params__y_feats]
        },
        {
            **dataset_params,
            'dataset_name':f'fedxgb_{args.federation}',
            'trees_per_client': args.n_estimators,
            'in_channels': 2,
            'oid': oid,
            'dataset_slice': 'test',
            'save_path':pathlib.Path(
                CFG_DATA_DIR / 
                'pth' / 
                f'fedxgb_{args.federation}'
            ),
        }
    )

    _worker_inference(
        pathlib.Path(
            CFG_DATA_DIR / 
            'pth' / 
            f'fedxgb_{args.federation}' / 
            f'cnn_test_dataset__n_estimators_{args.n_estimators}.X.client{oid}.parquet.gzip'
        ), 
        global_model.total_trees, 
        2
    )

2026-01-31 12:41:12.422 | DEBUG    | helper:fedxgbllr_cnn_create_tensor_dataset:502 - Memory report - function start - RSS=410204.7 MiB 
2026-01-31 12:41:12.572 | INFO     | helper:fedxgbllr_cnn_create_parquet_dataset:427 - Saving FedXGBllr test dataset to 
	data/pth/fedxgb_citi/cnn_test_dataset__n_estimators_37.X.client3460.02.parquet.gzip 
	data/pth/fedxgb_citi/cnn_test_dataset__n_estimators_37.y.client3460.02.parquet.gzip

[client #3460.02] test dataset labels: 100%|██████████| 2/2 [00:00<00:00,  9.23it/s]
2026-01-31 12:44:10.028 | DEBUG    | helper:fedxgbllr_cnn_create_tensor_dataset:510 - Memory report - function end - RSS=315936.4 MiB 
2026-01-31 12:44:10.029 | INFO     | helper:fedxgbllr_cnn_create_tensor_dataset:512 - Loading FedXGBllr test dataset from 
	 data/pth/fedxgb_citi/cnn_test_dataset__n_estimators_37.X.client3460.02.parquet.gzip 
	data/pth/fedxgb_citi/cnn_test_dataset__n_estimators_37.y.client3460.02.parquet.gzip

2026-01-31 12:44:10.029 | DEBUG    | helper:fedxgbllr_

# Save results

In [21]:
ebike_results = df_test.loc[
    :,
    [
        *dataset_params__y_feats,
        *[
            f'{feat}__global_fedxgb'
            for feat
            in dataset_params__y_feats
        ],
        *[
            f'{feat}__local_xgb'
            for feat
            in dataset_params__y_feats
        ]
    ]
].dropna(
    subset=[
        f'{dataset_params__y_feats[0]}__global_fedxgb'
    ]
)

In [ ]:
ebike_results.to_pickle(CFG_PICKLE_DIR / f'ebike_results_{args.federation}.pickle.gzip')